# Example 5 Wedding extended

## Mathematical Model:
$$
\begin{equation*}
	\textrm{max } z =  \sum_{i=1}^{n}\sum_{j=1}^{n}\sum_{t=1}^{T}s_{ij}x_{it}x_{jt}
\end{equation*}
$$

$$
\begin{align*}
	&& \textrm{subject to constraints: }  \sum_{i=1}^{n}x_{it} &\leq K_t && \textrm{for } t = 1,\dots,T \\
	&& \sum_{t=1}^{T}x_{it} &= 1 && \textrm{for }  i = 1,\dots,n \\
	&& x_{it} &\in \{0,1\} &&  \forall i,t \\
\end{align*}
$$


## Problem:
Nine people are to be distributed over three tables so that the sympathy values of the people per table are maximized.

### Sympathy/likeability values

|           | Anna | Berti | Carl | Dieter | Emil | Franz | Gerd | Hanna | Ilse |
|:---------:|:----:|:-----:|:----:|:------:|:----:|:-----:|:----:|:-----:|:----:|
|    Anna   |   0  |   6   |   4  |   10   |   1  |   2   |   6  |   2   |   4  |
|   Berti   |   6  |   0   |   7  |    3   |   6  |   1   |   5  |   10  |   5  |
|    Carl   |   4  |   7   |   0  |   10   |   6  |   3   |  10  |   1   |   4  |
|   Dieter  |  10  |   3   |  10  |    0   |   4  |   6   |   9  |   6   |  10  |
|    Emil   |   1  |   6   |   6  |    4   |   0  |   10  |   1  |   2   |   4  |
|   Franz   |   2  |   1   |   3  |    6   |  10  |   0   |   9  |   7   |   7  |
|    Gerd   |   6  |   5   |  10  |    9   |   1  |   9   |   0  |   1   |   9  |
|   Hanna   |   2  |   10  |   1  |    6   |   2  |   7   |   1  |   0   |   8  |
|    Ilse   |   4  |   5   |   4  |   10   |   4  |   7   |   9  |   8   |   0  |

### Table sizes
The three tables have the following sizes: 4, 3 and 2 seats


In [2]:
import gurobipy as gp
from gurobipy import GRB
import csv

## Sets - Definition

In [3]:
numberOfTables = 3
numberOfPersons = 9

tables = range(numberOfTables)
persons = range(numberOfPersons)

## Parameter definition

Sympathy values:

In [4]:
sympathy=[]
with open("Bsp4_SympathieWerte.csv", encoding="utf-8") as csv_file:
    csv_reader = csv.reader(csv_file)
    names = next(csv_reader)[1:]
    for row in csv_reader:
        rowAsInt = [int(item) for item in row[1:]]
        sympathy.append(rowAsInt)

sympathy

[[0, 6, 4, 10, 1, 2, 6, 2, 4],
 [6, 0, 7, 3, 6, 1, 5, 10, 5],
 [4, 7, 0, 10, 6, 3, 10, 1, 4],
 [10, 3, 10, 0, 4, 6, 9, 6, 10],
 [1, 6, 6, 4, 0, 10, 1, 2, 4],
 [2, 1, 3, 6, 10, 0, 9, 7, 7],
 [6, 5, 10, 9, 1, 9, 0, 1, 9],
 [2, 10, 1, 6, 2, 7, 1, 0, 8],
 [4, 5, 4, 10, 4, 7, 9, 8, 0]]

Table sizes:

In [5]:
capacity = [4, 3, 2]

## Initialising the model

In [6]:
model = gp.Model()

Restricted license - for non-production use only - expires 2025-11-24


## Initialising the variables

In [7]:
x = model.addVars(numberOfPersons, numberOfTables, vtype=GRB.BINARY, name="x")

## Defining the objective function

$$
\begin{equation*}
	\textrm{max } z =  \sum_{i=1}^{n}\sum_{j=1}^{n}\sum_{t=1}^{T}s_{ij}x_{it}x_{jt}
\end{equation*}
$$

In [8]:
z = gp.quicksum(sympathy[person1][person2] * x[person1, table] * x[person2, table]
                for table in tables for person1 in persons for person2 in persons if person1 != person2)    # if-condition removes unnecessary sum terms

## Adding constraints

$$
\begin{align*}
	&& \textrm{subject to constraints: }  \sum_{i=1}^{n}x_{it} &\leq K_t && \textrm{for } t = 1,\dots,T \\
	&& \sum_{t=1}^{T}x_{it} &= 1 && \textrm{for }  i = 1,\dots,n \\
\end{align*}
$$

In [9]:
model.addConstrs((x.sum('*',table) <= capacity[table] for table in tables), name="ConstrTable")

model.addConstrs((x.sum(person,'*') == 1 for person in persons), name="ConstrPerson")

{0: <gurobi.Constr *Awaiting Model Update*>,
 1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>,
 7: <gurobi.Constr *Awaiting Model Update*>,
 8: <gurobi.Constr *Awaiting Model Update*>}

## Optimization model

In [10]:
model.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) Ultra 5 125U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 12 rows, 27 columns and 54 nonzeros
Model fingerprint: 0x9aa5ee64
Variable types: 0 continuous, 27 integer (27 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 14 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%



## Result output

In [11]:
if model.Status == GRB.OPTIMAL:
    model.printAttr('ObjVal')
    model.printAttr('X')
elif model.Status == GRB.INFEASIBLE:
    print("Model is not solveable!")


      ObjVal 
------------
           0 

    Variable            X 
-------------------------
      x[0,2]            1 
      x[1,0]            1 
      x[2,2]            1 
      x[3,1]            1 
      x[4,0]            1 
      x[5,1]            1 
      x[6,0]            1 
      x[7,0]            1 
      x[8,1]            1 


In [12]:
model.write("model.lp")

In [13]:
model.write("loesung.sol")

## Intelligent output
Definition of the names:


In [14]:
# names = ["Anna", "Berti", "Carl", "Dieter", "Emil", "Franz", "Gerd", "Hanna", "Ilse"]

In [15]:
print("Distribution of seats\n")
for person in persons:
    for table in tables:
        if x[person, table].X > 0:
            print(f"{names[person]}\tsits at the table {table}")

Distribution of seats

 Anna	Sits at the table 2
 Berti	Sits at the table 0
 Carl	Sits at the table 2
 Dieter	Sits at the table 1
 Emil	Sits at the table 0
 Franz	Sits at the table 1
 Gerd	Sits at the table 0
 Hanna	Sits at the table 0
 Ilse	Sits at the table 1


In [16]:
for table in tables:
    print(f"\nPeople at the table {table}")
    for person1 in range(numberOfPersons-1):
        for person2 in range(person1+1, numberOfPersons):
            if x[person1, table].X * x[person2, table].X > 0:
                print(f"{names[person1]}\tand {names[person2]}\thave the sympathy value: {sympathy[person1][person2]}")


People at the table 0
 Berti	and  Emil	have the sympathy value: 6
 Berti	and  Gerd	have the sympathy value: 5
 Berti	and  Hanna	have the sympathy value: 10
 Emil	and  Gerd	have the sympathy value: 1
 Emil	and  Hanna	have the sympathy value: 2
 Gerd	and  Hanna	have the sympathy value: 1

People at the table 1
 Dieter	and  Franz	have the sympathy value: 6
 Dieter	and  Ilse	have the sympathy value: 10
 Franz	and  Ilse	have the sympathy value: 7

People at the table 2
 Anna	and  Carl	have the sympathy value: 4
